### Data Cleaning
All the issues or observations found in the data quality analysis will be resolved in this section.

#### Imports and Load

In [1]:
# Importing necessary libraries for the data cleaning and processing
import pandas as pd

df=pd.read_csv('../data/raw/ChargingRecords.csv')
df.shape

(72856, 13)

In [2]:
# Keeping a starting count
rows_start= len(df)
rows_start

72856

#### Step-1: Converting Date columns to datetime dtype

In [3]:
# The two required datetime columns are converted to datetime dtype.
df['StartDatetime']=pd.to_datetime(df['StartDatetime'])
df['EndDatetime']=pd.to_datetime(df['EndDatetime'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72856 entries, 0 to 72855
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   UserID          72856 non-null  int64         
 1   ChargerID       72856 non-null  int64         
 2   ChargerCompany  72856 non-null  int64         
 3   Location        72856 non-null  object        
 4   ChargerType     72856 non-null  int64         
 5   StartDay        72856 non-null  object        
 6   StartTime       72856 non-null  object        
 7   EndDay          72856 non-null  object        
 8   EndTime         72856 non-null  object        
 9   StartDatetime   72856 non-null  datetime64[ns]
 10  EndDatetime     72856 non-null  datetime64[ns]
 11  Duration        72856 non-null  int64         
 12  Demand          72856 non-null  float64       
dtypes: datetime64[ns](2), float64(1), int64(5), object(5)
memory usage: 7.2+ MB


#### Step-2: Removing the exact duplicates

In [4]:
# Counting No.of duplicates
df.duplicated().sum()

np.int64(13)

In [5]:
# Dropping the duplicates and checking the sum
df= df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

#### Step-3: Removing same session duplicates
The sessions having same user_id, charger_id and start datetime with different demand is considered as duplicates and is removed.

In [6]:
# Checking the duplicate count from the three below mnetioned columns
df.duplicated(subset=['UserID','ChargerID','StartDatetime']).sum()

np.int64(21)

In [7]:
# Removing the same session duplicates and checking the count
df= df.sort_values(['Demand','Duration'],ascending=False)
df= df.drop_duplicates(subset=['UserID','ChargerID','StartDatetime'],keep='first')
df= df.sort_index()
df.duplicated(subset=['UserID','ChargerID','StartDatetime']).sum()

np.int64(0)

#### Step-4: Removing negative duration sessions

In [8]:
# Checking the count of negative sessions
(df['Duration']<0).sum()

np.int64(17)

In [9]:
# Removing the negative duration sessions
df= df[df['Duration']>=0]
(df['Duration']<0).sum()

np.int64(0)

#### Step-4: Remove Zero Duration

In [10]:
# Checking the count of zero duration sessions
(df['Duration']==0).sum()

np.int64(709)

In [11]:
# Checking the % of non zero duration sessions 
1- (df['Duration']==0).sum()/len(df)

np.float64(0.9902616578531694)

In [12]:
# Checking the demand for the zero duration sessions. High demands are impossible for zero duration and can be removed.
df[df['Duration']==0]['Demand'].describe()

count    709.000000
mean       4.268900
std        9.532156
min        0.010000
25%        0.020000
50%        0.040000
75%        2.500000
max       62.400000
Name: Demand, dtype: float64

In [13]:
# Removing the zero duration sessions
df= df[df['Duration']>0]
(df['Duration']==0).sum()

np.int64(0)

In [14]:
df.shape

(72096, 13)

#### Step-5: Removing impossible power values

In [15]:
# Calculating the power
power= df['Demand']/(df['Duration']/60)
power.describe()

count    72096.000000
mean        11.656161
std         15.848680
min          0.000534
25%          6.077419
50%          6.700000
75%          7.080000
max       1657.800000
dtype: float64

In [16]:
# Checking the sessions with power > 350kwh
(power>350).sum()

np.int64(8)

In [17]:
# Checking which sessions have power >350kw
df[power>350][['Duration','Demand','Location']]

,Duration,Demand,Location
12039,2,29.07,public area
43770,5,43.74,camping
55133,4,45.04,market
55423,1,27.63,market
55426,4,31.73,market
55456,5,51.18,market
55466,6,47.50,market
63501,4,54.85,market


In [19]:
# Removing those 8 impossible Demand sessions
df= df[power<=350]
df.shape

C:\Users\VISHNU KASHYAP\AppData\Local\Temp\ipykernel_37356\2711155531.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df= df[power<=350]


(72088, 13)

In [21]:
# Confirming that the impossible demand sessions are removed
power= df['Demand']/(df['Duration']/60)
power.max()

np.float64(306.4)

#### Step-6: Drop redundant Date time columns

In [22]:
df= df.drop(columns=['StartDay','StartTime','EndDay','EndTime'])
df.shape

(72088, 9)

#### Step-7: Final checks
Checking whether all the changes made is reflected or not

In [27]:
# Nulls check
df.isnull().sum()

UserID            0
ChargerID         0
ChargerCompany    0
Location          0
ChargerType       0
StartDatetime     0
EndDatetime       0
Duration          0
Demand            0
dtype: int64

In [26]:
# Duplicates check
df.duplicated().sum(), df.duplicated(subset=['UserID','ChargerID','StartDatetime']).sum()

(np.int64(0), np.int64(0))

In [29]:
# Whole dataset duration check
df['StartDatetime'].min(), df['StartDatetime'].max()

(Timestamp('2021-09-30 17:14:00'), Timestamp('2022-09-30 23:11:00'))

In [32]:
# Checking the min duration with no zeros and negative durations
df[['Duration','Demand']].describe()

,Duration,Demand
count,72088.000000,72088.000000
mean,152.743688,17.569945
std,146.354670,13.457859
min,1.000000,0.010000
25%,41.000000,7.697500
50%,114.000000,14.260000
75%,203.000000,23.200000
max,1573.000000,97.000000


In [33]:
# Saving cleaned data to charging_clean .csv file
df.to_csv('../data/cleaned/charging_clean.csv', index=False)

In [34]:
# Checking the cleaned data shape and whether it'll reload properly or not
cleaned_data= pd.read_csv('../data/cleaned/charging_clean.csv')
cleaned_data.shape

(72088, 9)

#### Observations

1. Converted StartDatetime and EndDatetime from text to datetime. Both converted with no errors.
2. Removed 13 exact duplicate rows.
3. Removed 21 sessions that were recorded twice (same user, charger and start time). Kept the row with the higher Demand, because the other row was usually a failed plug-in.
4. Removed 17 rows with negative Duration (the end time was before the start time), so the record was wrong.
5. Removed 709 rows with zero Duration (under 1% of the data). Most had almost no energy (median Demand 0.04), so they look like failed attempts. Did not fill them with the median, because that would create charging time that never happened.
6. Removed 8 rows where the charging power was above 350 kW, which is not possible for any public charger. Most of these were at market locations, same as the negative Duration rows.
7. Dropped StartDay, StartTime, EndDay and EndTime. Because the datetime columns already have the same information.
8. Kept the IQR outliers in Demand and Duration, because long sessions and high energy use are possible.
9. Kept Duration as given. In about 2.6% of rows it doesn't match End − Start by 10 minutes or more, but we can't tell which value is wrong. This is noted as a limitation.
10. In total, 768 rows were removed (about 1.05%). The cleaned data has 72,088 rows and 9 columns, saved as data/cleaned/charging_clean.csv.